# MoveBreak Iteration 3 — Hourly Footfall Prediction Model

This notebook builds a reproducible **rolling one-hour-ahead pedestrian footfall model** for selected City of Melbourne sensors. It downloads and audits the official data, links each pedestrian counter to its nearest microclimate station, creates leakage-controlled time-series features, compares a seasonal baseline with two machine-learning models, evaluates on the final 28 days, and exports the fitted model and predictions.

**Official sources**

1. [Pedestrian Counting System — counts per hour](https://data.melbourne.vic.gov.au/explore/dataset/pedestrian-counting-system-monthly-counts-per-hour/) — prediction target.
2. [Pedestrian Counting System — sensor locations](https://data.melbourne.vic.gov.au/explore/dataset/pedestrian-counting-system-sensor-locations/) — pedestrian sensor coordinates.
3. [Microclimate sensors data](https://data.melbourne.vic.gov.au/explore/dataset/microclimate-sensors-data/) — air temperature and station coordinates.

The notebook validates the **live API schema and metadata before modelling**. It never treats missing source hours as zero. The default six-month window is long enough for weekly lags and a meaningful time holdout; use `QUICK_RUN = True` for a short pipeline test, not for final model claims.

### Modelling contract

- Target: pedestrian count for one sensor and one local clock hour.
- Evaluation: rolling one-hour-ahead backtest; prior test-period observations may be used as lags, as they would be in an hourly production refresh.
- Available features: calendar variables, recent pedestrian counts, and the latest microclimate temperature at or before the target hour.
- Not included: personal preference or `localStorage` history. Those belong in the downstream recommendation/ranking layer.
- Limitation: observed temperature is used for historical evaluation. A true future forecast must replace it with weather known at the prediction issue time.

In [ ]:
# Uncomment and run once if your environment is missing packages.
# %pip install pandas numpy matplotlib scikit-learn joblib

import json
import platform
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

API_ROOT = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets"
DATASETS = {
    "footfall": "pedestrian-counting-system-monthly-counts-per-hour",
    "locations": "pedestrian-counting-system-sensor-locations",
    "microclimate": "microclimate-sensors-data",
}
SOURCE_URLS = {
    key: f"https://data.melbourne.vic.gov.au/explore/dataset/{dataset_id}/"
    for key, dataset_id in DATASETS.items()
}

# Final run: six months. Quick run only confirms code flow and is too short for final claims.
QUICK_RUN = False
START_DATE = "2026-03-01"
END_DATE_EXCLUSIVE = "2026-09-01" if not QUICK_RUN else "2026-04-15"
SENSOR_IDS = [17, 19, 79]
TEST_DAYS = 28 if not QUICK_RUN else 7
MAX_WEATHER_DISTANCE_M = 500
MAX_WEATHER_AGE_MIN = 60
PAGE_SIZE = 100
MAX_DAYS = 220

OUTPUT_DIR = Path("movebreak_footfall_model_outputs")
CACHE_DIR = OUTPUT_DIR / "api_cache"
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

DAYS = pd.date_range(
    START_DATE,
    pd.Timestamp(END_DATE_EXCLUSIVE) - pd.Timedelta(days=1),
    freq="D",
)
assert 35 <= len(DAYS) <= MAX_DAYS, "Use 35–220 days so lag features and the holdout are meaningful."
assert TEST_DAYS >= 7 and TEST_DAYS < len(DAYS) - 8

print(f"Window: {START_DATE} to {END_DATE_EXCLUSIVE} (exclusive), {len(DAYS)} days")
print("Sensors:", SENSOR_IDS)
print("Versions:", {"python": platform.python_version(), "pandas": pd.__version__,
                    "scikit_learn": sklearn.__version__})

## 1. Verify live source metadata and schema

The portal is the authority for column meanings. This cell stores the metadata returned during the run and stops if required fields disappear. It also resolves known field-name variants explicitly instead of silently guessing from numeric columns.

In [ ]:
def api_request(dataset_key, suffix="", params=None, attempts=5):
    endpoint = f"{API_ROOT}/{DATASETS[dataset_key]}{suffix}"
    if params:
        endpoint += "?" + urllib.parse.urlencode(params)
    for attempt in range(attempts):
        try:
            request = urllib.request.Request(
                endpoint,
                headers={"Accept": "application/json", "User-Agent": "MoveBreak-coursework-model/1.0"},
            )
            with urllib.request.urlopen(request, timeout=90) as response:
                return json.load(response)
        except urllib.error.HTTPError as exc:
            retryable = exc.code in {429, 500, 502, 503, 504}
            if not retryable or attempt == attempts - 1:
                raise RuntimeError(f"{dataset_key}: HTTP {exc.code}; {endpoint}") from exc
        except Exception:
            if attempt == attempts - 1:
                raise
        time.sleep(min(2 ** attempt, 15))


metadata = {key: api_request(key) for key in DATASETS}
samples = {
    key: api_request(key, "/records", {"limit": 3})["results"]
    for key in DATASETS
}
assert all(samples.values()), "At least one official source returned no sample records."

required = {
    "footfall": {"sensing_date", "hourday", "location_id"},
    "microclimate": {"received_at", "airtemperature", "latlong"},
}
for key, required_fields in required.items():
    missing = required_fields - set(samples[key][0])
    assert not missing, f"{key} schema changed; missing {sorted(missing)}"

COUNT_FIELD = next(
    (name for name in ["pedestriancount", "hourly_counts", "hourly_count",
                       "total_of_directions", "pedestrian_count", "total_count", "count"]
     if name in samples["footfall"][0]),
    None,
)
LOCATION_SENSOR_FIELD = next(
    (name for name in ["location_id", "sensor_id"] if name in samples["locations"][0]),
    None,
)
assert COUNT_FIELD, f"Select the count field after inspecting: {sorted(samples['footfall'][0])}"
assert LOCATION_SENSOR_FIELD, f"Select the location ID field after inspecting: {sorted(samples['locations'][0])}"

source_audit = pd.DataFrame([
    {
        "source": key,
        "dataset_id": DATASETS[key],
        "portal_url": SOURCE_URLS[key],
        "sample_columns": ", ".join(sorted(samples[key][0])),
        "live_title": metadata[key].get("metas", {}).get("default", {}).get("title"),
    }
    for key in DATASETS
])
display(source_audit)
print("Resolved count field:", COUNT_FIELD)
print("Resolved location ID field:", LOCATION_SENSOR_FIELD)

(OUTPUT_DIR / "source_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
source_audit.to_csv(OUTPUT_DIR / "source_audit.csv", index=False)

## 2. Download complete daily slices with restartable caching

Daily pagination keeps every query below the API's offset ceiling. Each slice records the server-reported total and downloaded row count. The model cannot proceed if a slice is truncated. Microclimate starts one UTC date earlier so a Melbourne-local midnight can still find a prior observation.

In [ ]:
def fetch_day(dataset_key, day):
    stamp = pd.Timestamp(day).strftime("%Y-%m-%d")
    following = (pd.Timestamp(day) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    date_field = "sensing_date" if dataset_key == "footfall" else "received_at"
    where = f"{date_field} >= date'{stamp}' AND {date_field} < date'{following}'"
    cache_path = CACHE_DIR / f"{dataset_key}_{stamp}.json"

    cache_hit = cache_path.exists()
    if cache_hit:
        payload = json.loads(cache_path.read_text(encoding="utf-8"))
        assert payload["where"] == where, f"Cached filter mismatch: {cache_path}"
        rows, total = payload["rows"], payload["total"]
    else:
        rows, total, offset = [], None, 0
        while True:
            payload = api_request(
                dataset_key,
                "/records",
                {"where": where, "limit": PAGE_SIZE, "offset": offset,
                 "order_by": f"{date_field} ASC"},
            )
            if total is None:
                total = int(payload["total_count"])
                assert total < 10_000, f"Split {dataset_key} {stamp} into smaller windows."
            part = payload["results"]
            if not part and offset < total:
                raise RuntimeError(f"{dataset_key} {stamp}: pagination stopped at {offset}/{total}")
            rows.extend(part)
            offset += len(part)
            if offset >= total:
                break
            time.sleep(0.05)
        assert len(rows) == total
        cache_path.write_text(
            json.dumps({"where": where, "total": total, "rows": rows}), encoding="utf-8"
        )

    return rows, {
        "source": dataset_key,
        "day": stamp,
        "reported_total": total,
        "downloaded": len(rows),
        "complete": len(rows) == total,
        "from_cache": cache_hit,
    }


raw = {}
download_audit_rows = []
for dataset_key in ["footfall", "microclimate"]:
    dataset_days = DAYS if dataset_key == "footfall" else DAYS.insert(0, DAYS[0] - pd.Timedelta(days=1))
    batches = []
    for index, day in enumerate(dataset_days, start=1):
        rows, audit_row = fetch_day(dataset_key, day)
        batches.extend(rows)
        download_audit_rows.append(audit_row)
        if index == 1 or index % 14 == 0 or index == len(dataset_days):
            print(f"{dataset_key}: {index}/{len(dataset_days)} days; latest={audit_row['day']}")
    raw[dataset_key] = pd.DataFrame(batches)

download_audit = pd.DataFrame(download_audit_rows)
display(download_audit.groupby("source").agg(
    days=("day", "count"),
    reported_rows=("reported_total", "sum"),
    downloaded_rows=("downloaded", "sum"),
    complete_days=("complete", "sum"),
))
download_audit.to_csv(OUTPUT_DIR / "daily_download_audit.csv", index=False)
assert download_audit["complete"].all(), "At least one source-day is incomplete."
assert not raw["footfall"].empty and not raw["microclimate"].empty
print("Raw rows:", {key: len(value) for key, value in raw.items()})

## 3. Clean pedestrian targets and sensor coordinates

The pedestrian date and hour describe Melbourne local clock time. The source cannot uniquely represent a repeated daylight-saving hour, so ambiguous/nonexistent clock hours are removed and reported. Duplicate sensor-hours stop the notebook rather than being averaged.

In [ ]:
def parse_point(value):
    if isinstance(value, dict):
        if {"lat", "lon"} <= set(value):
            return float(value["lat"]), float(value["lon"])
        if value.get("type") == "Point" and len(value.get("coordinates", [])) >= 2:
            lon, lat = value["coordinates"][:2]
            return float(lat), float(lon)
    return np.nan, np.nan


footfall = raw["footfall"].copy()
footfall["sensor_id"] = pd.to_numeric(footfall["location_id"], errors="coerce").astype("Int64")
footfall = footfall[footfall["sensor_id"].isin(SENSOR_IDS)].copy()
footfall["count"] = pd.to_numeric(footfall[COUNT_FIELD], errors="coerce")
hours = pd.to_numeric(footfall["hourday"], errors="coerce")
assert hours.dropna().between(0, 23).all(), "Unexpected hourday values."
local_naive = (
    pd.to_datetime(footfall["sensing_date"].astype("string").str[:10], errors="coerce")
    + pd.to_timedelta(hours, unit="h")
)
footfall["time_local"] = local_naive.dt.tz_localize(
    "Australia/Melbourne", ambiguous="NaT", nonexistent="NaT"
)
target_quality = {
    "selected_raw_rows": len(footfall),
    "invalid_or_dst_ambiguous_time": int(footfall["time_local"].isna().sum()),
    "missing_count": int(footfall["count"].isna().sum()),
    "negative_count": int((footfall["count"] < 0).sum()),
}
footfall = footfall.dropna(subset=["sensor_id", "time_local", "count"])
footfall = footfall[footfall["count"] >= 0].copy()
assert not footfall.duplicated(["sensor_id", "time_local"]).any(), "Duplicate sensor-hours found."

# Download the complete location reference table.
location_rows, offset, location_total = [], 0, None
while location_total is None or offset < location_total:
    payload = api_request("locations", "/records", {"limit": PAGE_SIZE, "offset": offset})
    if location_total is None:
        location_total = int(payload["total_count"])
    part = payload["results"]
    assert part or offset >= location_total, "Location pagination ended early."
    location_rows.extend(part)
    offset += len(part)
assert len(location_rows) == location_total

locations = pd.DataFrame(location_rows)
locations["sensor_id"] = pd.to_numeric(
    locations[LOCATION_SENSOR_FIELD], errors="coerce"
).astype("Int64")
locations = locations[locations["sensor_id"].isin(SENSOR_IDS)].copy()
if {"latitude", "longitude"} <= set(locations.columns):
    locations["lat"] = pd.to_numeric(locations["latitude"], errors="coerce")
    locations["lon"] = pd.to_numeric(locations["longitude"], errors="coerce")
else:
    point_field = next(
        (name for name in ["location", "geo_point_2d", "geometry"] if name in locations.columns),
        None,
    )
    assert point_field, f"No supported coordinate field in {sorted(locations.columns)}"
    parsed = locations[point_field].map(parse_point)
    locations["lat"] = parsed.map(lambda item: item[0])
    locations["lon"] = parsed.map(lambda item: item[1])

assert not locations["sensor_id"].duplicated().any()
assert set(SENSOR_IDS) <= set(locations["sensor_id"].dropna().astype(int))
assert locations["lat"].between(-38, -37).all() and locations["lon"].between(144, 146).all()

target_quality["clean_sensor_hours"] = len(footfall)
target_quality["zero_counts"] = int((footfall["count"] == 0).sum())
display(pd.DataFrame([target_quality]))
display(locations[["sensor_id", "lat", "lon"]])

## 4. Link the nearest microclimate site and make a past-only join

Each pedestrian sensor is linked to the nearest coordinate observed in the microclimate feed. Only links within 500 metres are accepted. `merge_asof(..., direction="backward")` guarantees that a future weather reading is never attached to an earlier footfall target.

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    lat1_rad, lat2_rad = np.radians(lat1), np.radians(lat2)
    dlat = lat2_rad - lat1_rad
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    return 12_742_000 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


micro = raw["microclimate"].copy()
micro["climate_time"] = pd.to_datetime(micro["received_at"], errors="coerce", utc=True).dt.tz_convert(
    "Australia/Melbourne"
)
micro["temperature"] = pd.to_numeric(micro["airtemperature"], errors="coerce")
parsed = micro["latlong"].map(parse_point)
micro["lat"] = parsed.map(lambda item: item[0])
micro["lon"] = parsed.map(lambda item: item[1])
micro = micro.dropna(subset=["climate_time", "temperature", "lat", "lon"])
micro = micro[micro["lat"].between(-38, -37) & micro["lon"].between(144, 146)].copy()
assert not micro.empty

climate_sites = micro[["lat", "lon"]].drop_duplicates().reset_index(drop=True)
links = []
for sensor in locations.itertuples():
    distances = haversine_m(
        sensor.lat, sensor.lon, climate_sites["lat"].to_numpy(), climate_sites["lon"].to_numpy()
    )
    nearest = int(np.argmin(distances))
    links.append({
        "sensor_id": int(sensor.sensor_id),
        "climate_lat": float(climate_sites.at[nearest, "lat"]),
        "climate_lon": float(climate_sites.at[nearest, "lon"]),
        "distance_m": float(distances[nearest]),
    })
links = pd.DataFrame(links)
links["accepted"] = links["distance_m"] <= MAX_WEATHER_DISTANCE_M
display(links)
assert links["accepted"].all(), "A selected sensor has no microclimate site within the allowed radius."

targets = footfall[["sensor_id", "time_local", "count"]].merge(
    links, on="sensor_id", validate="many_to_one"
)
joined_parts = []
for (lat, lon), group in targets.groupby(["climate_lat", "climate_lon"]):
    station_weather = (
        micro[(micro["lat"] == lat) & (micro["lon"] == lon)][["climate_time", "temperature"]]
        .sort_values("climate_time")
        .drop_duplicates("climate_time", keep="last")
    )
    joined_parts.append(pd.merge_asof(
        group.sort_values("time_local"),
        station_weather,
        left_on="time_local",
        right_on="climate_time",
        direction="backward",
        tolerance=pd.Timedelta(minutes=MAX_WEATHER_AGE_MIN),
    ))

model_base = pd.concat(joined_parts, ignore_index=True).sort_values(["sensor_id", "time_local"])
model_base["weather_age_min"] = (
    model_base["time_local"] - model_base["climate_time"]
).dt.total_seconds() / 60
assert not model_base.duplicated(["sensor_id", "time_local"]).any()
assert model_base["weather_age_min"].dropna().between(0, MAX_WEATHER_AGE_MIN).all()

coverage = model_base.groupby("sensor_id").agg(
    target_hours=("count", "size"),
    weather_matches=("temperature", "count"),
    start=("time_local", "min"),
    end=("time_local", "max"),
    median_count=("count", "median"),
    median_weather_age_min=("weather_age_min", "median"),
).reset_index()
coverage["weather_match_pct"] = 100 * coverage["weather_matches"] / coverage["target_hours"]
display(coverage)
assert coverage["weather_match_pct"].min() >= 80, "Weather coverage below 80%; inspect before modelling."

links.to_csv(OUTPUT_DIR / "sensor_climate_links.csv", index=False)
coverage.to_csv(OUTPUT_DIR / "joined_data_coverage.csv", index=False)

## 5. Exploratory checks before feature engineering

The plots are diagnostics, not proof of causality. Hour-of-week profiles should show a stable weekly rhythm; the temperature scatter only shows association and is likely confounded by time of day and season.

In [ ]:
eda = model_base.copy()
eda["hour"] = eda["time_local"].dt.hour
eda["day_of_week"] = eda["time_local"].dt.dayofweek
eda["hour_of_week"] = eda["day_of_week"] * 24 + eda["hour"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for sensor_id, group in eda.groupby("sensor_id"):
    profile = group.groupby("hour_of_week")["count"].median()
    axes[0].plot(profile.index, profile.values, label=f"Sensor {sensor_id}")
axes[0].set(title="Median weekly pedestrian pattern", xlabel="Hour of week (Monday = 0)",
            ylabel="Median hourly count")
axes[0].legend()
axes[0].grid(alpha=0.2)

sample_for_scatter = eda.dropna(subset=["temperature"]).sample(
    n=min(3000, eda["temperature"].notna().sum()), random_state=RANDOM_SEED
)
axes[1].scatter(sample_for_scatter["temperature"], sample_for_scatter["count"], s=8, alpha=0.2)
axes[1].set(title="Temperature and footfall (descriptive only)", xlabel="Air temperature",
            ylabel="Hourly count")
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

display(eda.groupby("sensor_id")["count"].describe()[["count", "mean", "50%", "std", "max"]])

## 6. Create leakage-controlled features

Every count-derived predictor uses an exact earlier timestamp or a left-closed time window. Missing source hours therefore remain missing instead of making row-based lags silently drift. `rolling_mean_24` and `rolling_mean_168` exclude the target hour. Dropping rows without a full weekly lag makes the comparison with the 168-hour seasonal baseline fair.

In [ ]:
features = model_base.sort_values(["sensor_id", "time_local"]).copy()
features["sensor_id_cat"] = features["sensor_id"].astype("string")
features["hour"] = features["time_local"].dt.hour
features["day_of_week"] = features["time_local"].dt.dayofweek
features["month"] = features["time_local"].dt.month
features["is_weekend"] = (features["day_of_week"] >= 5).astype(int)
features["hour_sin"] = np.sin(2 * np.pi * features["hour"] / 24)
features["hour_cos"] = np.cos(2 * np.pi * features["hour"] / 24)
features["dow_sin"] = np.sin(2 * np.pi * features["day_of_week"] / 7)
features["dow_cos"] = np.cos(2 * np.pi * features["day_of_week"] / 7)
features["month_sin"] = np.sin(2 * np.pi * features["month"] / 12)
features["month_cos"] = np.cos(2 * np.pi * features["month"] / 12)

# Build exact elapsed-time lags in UTC. If an exact source hour is absent, the lag remains NaN.
features["time_utc"] = features["time_local"].dt.tz_convert("UTC")
for lag_hours in [1, 24, 168]:
    lag_name = f"count_lag_{lag_hours}"
    lag_lookup = features[["sensor_id", "time_utc", "count"]].copy()
    lag_lookup["time_utc"] = lag_lookup["time_utc"] + pd.Timedelta(hours=lag_hours)
    lag_lookup = lag_lookup.rename(columns={"count": lag_name})
    features = features.merge(
        lag_lookup, on=["sensor_id", "time_utc"], how="left", validate="one_to_one"
    )

# Time-window means are left-closed, so the target hour is never included.
features["rolling_mean_24"] = np.nan
features["rolling_mean_168"] = np.nan
for sensor_id, group in features.groupby("sensor_id", sort=False):
    ordered = group.sort_values("time_utc")
    history = ordered.set_index("time_utc")["count"]
    features.loc[ordered.index, "rolling_mean_24"] = history.rolling(
        "24h", closed="left", min_periods=18
    ).mean().to_numpy()
    features.loc[ordered.index, "rolling_mean_168"] = history.rolling(
        "168h", closed="left", min_periods=120
    ).mean().to_numpy()

feature_columns = [
    "sensor_id_cat", "temperature", "weather_age_min",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos", "is_weekend",
    "count_lag_1", "count_lag_24", "count_lag_168", "rolling_mean_24", "rolling_mean_168",
]
required_model_columns = feature_columns + ["count", "time_local", "sensor_id"]
model_data = features.dropna(subset=required_model_columns).copy()
assert len(model_data) >= 1000, "Too few complete rows; use a longer window or inspect missingness."
assert (model_data["weather_age_min"] >= 0).all(), "Future weather leakage detected."

feature_audit = pd.DataFrame({
    "column": required_model_columns,
    "missing_before_drop": [int(features[column].isna().sum()) for column in required_model_columns],
    "dtype": [str(features[column].dtype) for column in required_model_columns],
})
display(feature_audit)
print(f"Model-ready rows: {len(model_data):,} of {len(features):,}")
model_data.to_csv(OUTPUT_DIR / "model_ready_data.csv", index=False)
feature_audit.to_csv(OUTPUT_DIR / "feature_audit.csv", index=False)

## 7. Time-based train/test split

The final `TEST_DAYS` calendar days form the holdout. No random shuffle is used. This is a rolling backtest: lag values inside the test period use counts observed before each target hour.

In [ ]:
latest_time = model_data["time_local"].max()
cutoff = latest_time.floor("D") - pd.Timedelta(days=TEST_DAYS - 1)
train = model_data[model_data["time_local"] < cutoff].copy()
test = model_data[model_data["time_local"] >= cutoff].copy()

assert not train.empty and not test.empty
assert train["time_local"].max() < test["time_local"].min()
assert set(test["sensor_id"].unique()) <= set(train["sensor_id"].unique())
assert len(train) >= 2 * len(test), "Training period is too short relative to the holdout."

X_train, y_train = train[feature_columns], train["count"]
X_test, y_test = test[feature_columns], test["count"]
print("Train:", train["time_local"].min(), "to", train["time_local"].max(), "rows=", len(train))
print("Test: ", test["time_local"].min(), "to", test["time_local"].max(), "rows=", len(test))
print("Cutoff:", cutoff)

## 8. Train baseline, Ridge, and Random Forest models

- **Seasonal naive**: same sensor and clock hour one week earlier (`count_lag_168`). This is the minimum credible baseline for strongly weekly footfall data.
- **Ridge**: regularised linear model after imputation, scaling, and sensor one-hot encoding.
- **Random Forest**: nonlinear model that captures interactions among time, recent demand, sensor, and temperature.

All hyperparameters are fixed before looking at the holdout. If tuning is added later, use a second time-based validation window inside the training period—not the test set.

In [ ]:
categorical_features = ["sensor_id_cat"]
numeric_features = [column for column in feature_columns if column not in categorical_features]

try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # scikit-learn < 1.2
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

ridge_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("sensor", one_hot, categorical_features),
])

try:
    forest_one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    forest_one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

forest_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ("sensor", forest_one_hot, categorical_features),
])

models = {
    "Ridge": Pipeline([
        ("preprocess", ridge_preprocessor),
        ("model", Ridge(alpha=10.0)),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", forest_preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=3,
            max_features=0.8,
            n_jobs=-1,
            random_state=RANDOM_SEED,
        )),
    ]),
}

for name, estimator in models.items():
    started = time.time()
    estimator.fit(X_train, y_train)
    print(f"Fitted {name} in {time.time() - started:.1f}s")

## 9. Evaluate on the untouched final window

MAE is the primary metric because it is interpretable as pedestrians per sensor-hour. RMSE emphasises large misses. WAPE normalises total absolute error by total observed footfall. R² is included as a secondary diagnostic. Predictions are clipped at zero because pedestrian counts cannot be negative.

In [ ]:
def metric_row(name, actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    absolute_error = np.abs(actual - predicted)
    denominator = np.abs(actual).sum()
    return {
        "model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "WAPE_pct": 100 * absolute_error.sum() / denominator if denominator else np.nan,
        "R2": r2_score(actual, predicted),
    }


prediction_frame = test[["time_local", "sensor_id", "count"]].reset_index(drop=True)
prediction_frame["Seasonal naive"] = test["count_lag_168"].to_numpy()
metrics = [metric_row("Seasonal naive", y_test, prediction_frame["Seasonal naive"])]

for name, estimator in models.items():
    prediction_frame[name] = np.clip(estimator.predict(X_test), 0, None)
    metrics.append(metric_row(name, y_test, prediction_frame[name]))

metrics = pd.DataFrame(metrics).sort_values("MAE").reset_index(drop=True)
display(metrics.round({"MAE": 2, "RMSE": 2, "WAPE_pct": 2, "R2": 3}))

ml_names = list(models)
best_model_name = min(ml_names, key=lambda name: float(metrics.loc[metrics["model"] == name, "MAE"].iloc[0]))
best_model = models[best_model_name]
prediction_frame["prediction"] = prediction_frame[best_model_name]
prediction_frame["absolute_error"] = np.abs(prediction_frame["count"] - prediction_frame["prediction"])

by_sensor = []
for sensor_id, group in prediction_frame.groupby("sensor_id"):
    for name in ["Seasonal naive"] + ml_names:
        row = metric_row(name, group["count"], group[name])
        row["sensor_id"] = sensor_id
        by_sensor.append(row)
by_sensor = pd.DataFrame(by_sensor)
display(by_sensor.sort_values(["sensor_id", "MAE"]))

baseline_mae = float(metrics.loc[metrics["model"] == "Seasonal naive", "MAE"].iloc[0])
best_ml_mae = float(metrics.loc[metrics["model"] == best_model_name, "MAE"].iloc[0])
improvement_pct = 100 * (baseline_mae - best_ml_mae) / baseline_mae
print(f"Best ML model: {best_model_name}")
print(f"MAE change versus seasonal baseline: {improvement_pct:+.2f}%")
if improvement_pct <= 0:
    print("Gate not passed: the ML model does not beat the seasonal baseline on MAE.")
else:
    print("Gate passed: the ML model beats the seasonal baseline on MAE.")

## 10. Prediction diagnostics and empirical uncertainty band

The interval below is an empirical 80% error band estimated separately for each sensor from the holdout residuals. It is descriptive, not a formal probabilistic forecast, and should be recalibrated on new data.

In [ ]:
sensor_error_band = (
    prediction_frame.groupby("sensor_id")["absolute_error"].quantile(0.80).rename("abs_error_q80")
)
prediction_frame = prediction_frame.merge(sensor_error_band, on="sensor_id", how="left")
prediction_frame["prediction_lower_80"] = np.clip(
    prediction_frame["prediction"] - prediction_frame["abs_error_q80"], 0, None
)
prediction_frame["prediction_upper_80"] = prediction_frame["prediction"] + prediction_frame["abs_error_q80"]

sensor_ids = sorted(prediction_frame["sensor_id"].unique())
fig, axes = plt.subplots(len(sensor_ids), 1, figsize=(15, 3.6 * len(sensor_ids)), sharex=True)
axes = np.atleast_1d(axes)
plot_start = prediction_frame["time_local"].max() - pd.Timedelta(days=7)
for axis, sensor_id in zip(axes, sensor_ids):
    plot_data = prediction_frame[
        (prediction_frame["sensor_id"] == sensor_id) & (prediction_frame["time_local"] >= plot_start)
    ]
    axis.plot(plot_data["time_local"], plot_data["count"], label="Actual", linewidth=1.5)
    axis.plot(plot_data["time_local"], plot_data["prediction"], label=best_model_name, linewidth=1.2)
    axis.fill_between(
        plot_data["time_local"], plot_data["prediction_lower_80"], plot_data["prediction_upper_80"],
        alpha=0.15, label="Empirical 80% band",
    )
    axis.set_title(f"Sensor {sensor_id}: final seven test days")
    axis.set_ylabel("Pedestrians/hour")
    axis.grid(alpha=0.2)
    axis.legend(loc="upper right")
axes[-1].set_xlabel("Melbourne local time")
plt.tight_layout()
plt.show()

error_by_hour = prediction_frame.assign(hour=prediction_frame["time_local"].dt.hour).groupby("hour").agg(
    MAE=("absolute_error", "mean"),
    actual_mean=("count", "mean"),
    rows=("count", "size"),
)
display(error_by_hour)
error_by_hour["MAE"].plot(kind="bar", figsize=(12, 3.5), title=f"{best_model_name} MAE by hour")
plt.ylabel("MAE")
plt.tight_layout()
plt.show()

## 11. Model interpretation with permutation importance

Permutation importance measures the increase in test MAE when one original input column is shuffled. It describes predictive reliance, not causality. Correlated lag features can share importance.

In [ ]:
importance_rows = test.sample(n=min(3000, len(test)), random_state=RANDOM_SEED)
importance = permutation_importance(
    best_model,
    importance_rows[feature_columns],
    importance_rows["count"],
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance_mean_MAE_increase": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean_MAE_increase", ascending=False)
display(feature_importance)

plot_importance = feature_importance.sort_values("importance_mean_MAE_increase")
plt.figure(figsize=(9, 5.5))
plt.barh(plot_importance["feature"], plot_importance["importance_mean_MAE_increase"])
plt.xlabel("Increase in MAE after permutation")
plt.title(f"Permutation importance — {best_model_name}")
plt.tight_layout()
plt.show()

## 12. Refit, export, and provide an inference helper

After evaluation, the selected ML pipeline is refitted on all model-ready rows for deployment. The helper expects the same engineered feature columns. In production, construct them from the latest pedestrian history and weather available at the forecast issue time. Do not substitute browser `localStorage` values for these model inputs; use those values later to personalise or re-rank recommendations.

In [ ]:
final_model = models[best_model_name]
final_model.fit(model_data[feature_columns], model_data["count"])

def predict_footfall(feature_rows, fitted_model=final_model):
    """Return non-negative hourly predictions for rows with the documented feature schema."""
    missing = set(feature_columns) - set(feature_rows.columns)
    if missing:
        raise ValueError(f"Missing feature columns: {sorted(missing)}")
    output = feature_rows.copy()
    output["predicted_footfall"] = np.clip(
        fitted_model.predict(output[feature_columns]), 0, None
    )
    return output


# Smoke-test the serialized inference contract before saving.
smoke_input = model_data.tail(5)[feature_columns].copy()
smoke_predictions = predict_footfall(smoke_input)
assert len(smoke_predictions) == 5
assert np.isfinite(smoke_predictions["predicted_footfall"]).all()
assert (smoke_predictions["predicted_footfall"] >= 0).all()

artifact = {
    "model": final_model,
    "feature_columns": feature_columns,
    "model_name": best_model_name,
    "sensor_ids": SENSOR_IDS,
    "training_window": {"start": START_DATE, "end_exclusive": END_DATE_EXCLUSIVE},
    "timezone": "Australia/Melbourne",
    "target": "hourly pedestrian count",
}
joblib.dump(artifact, OUTPUT_DIR / "footfall_model.joblib")

metadata_export = {
    "model_name": best_model_name,
    "feature_columns": feature_columns,
    "sensor_ids": SENSOR_IDS,
    "start_date": START_DATE,
    "end_date_exclusive": END_DATE_EXCLUSIVE,
    "test_days": TEST_DAYS,
    "best_ml_mae": best_ml_mae,
    "seasonal_baseline_mae": baseline_mae,
    "mae_improvement_vs_baseline_pct": improvement_pct,
    "sources": SOURCE_URLS,
    "random_seed": RANDOM_SEED,
    "versions": {"pandas": pd.__version__, "scikit_learn": sklearn.__version__},
    "evaluation_mode": "rolling one-hour-ahead",
}
(OUTPUT_DIR / "model_metadata.json").write_text(
    json.dumps(metadata_export, indent=2), encoding="utf-8"
)
metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)
by_sensor.to_csv(OUTPUT_DIR / "metrics_by_sensor.csv", index=False)
prediction_frame.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

loaded = joblib.load(OUTPUT_DIR / "footfall_model.joblib")
round_trip = np.clip(loaded["model"].predict(smoke_input[loaded["feature_columns"]]), 0, None)
assert np.allclose(round_trip, smoke_predictions["predicted_footfall"])

print("Exported files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(" -", path)
print("\nModel artifact round-trip test passed.")

## 13. Decision gates for the Iteration 3 presentation

Report the model only after checking all of these:

1. Every row in `daily_download_audit.csv` has `complete=True`.
2. All selected sensors have a microclimate link within 500 m and at least 80% matched weather coverage.
3. The holdout is strictly later than the training window.
4. The seasonal naive baseline is shown beside the ML models.
5. Claim an improvement only if the chosen ML model beats the baseline on holdout MAE.
6. Describe this as a rolling one-hour-ahead experiment, not a long-range forecast.
7. Do not claim temperature causes footfall changes; it is a contextual predictor.
8. For deployment, replace observed test-period temperature with weather available at prediction time.
9. Retrain and re-evaluate when adding sensors or extending to another season.

The exported `test_predictions.csv`, `model_metrics.csv`, and `feature_importance.csv` are presentation-ready evidence. `footfall_model.joblib` is the fitted pipeline for a Python backend. Browser `localStorage` can separately remember user choices and help re-rank break locations, but it does not retrain this model.